# Universal Context Engine — Gradio Standalone UI

Copyright 2025-2026, Denis Rothman

---

## What This Notebook Is

This notebook is the **deployable evolution** of the Universal Context Engine UI. While the companion notebook (`Universal_Context_Engine_UI.ipynb`) demonstrated an IPython widget interface, this version replaces it with a **[Gradio](https://www.gradio.app/)** web application that can be:

- Run interactively inside Google Colab with a live public URL
- Launched locally on any machine as a standalone web server
- Deployed to Hugging Face Spaces or any container platform with a single command

The engine underneath is **identical** — the same Glass Box, zero-business-logic architecture described in *Context Engineering for Multi-Agent Systems*. Only the presentation layer changes.

---

## Core Thesis Reminder

> **The architecture is domain-agnostic.**

The engine contains **zero hard-coded business rules**. It adapts entirely to:
1. The **context** retrieved from the vector store (Pinecone)
2. The **goal** provided through the Control Deck

Running the exact same code on Legal documents or Marketing copy produces completely different, domain-appropriate outputs. This notebook proves that claim through a live, interactive demonstration.

---

## Why Gradio?

| Feature | ipywidgets (previous UI) | Gradio (this notebook) |
|---|---|---|
| Runs in Colab | ✅ | ✅ |
| Public shareable URL | ❌ | ✅ (auto-generated) |
| Deployable standalone | ❌ | ✅ |
| Mobile-friendly | ❌ | ✅ |
| Custom themes | Limited | ✅ |
| Hugging Face Spaces ready | ❌ | ✅ |

---

## Prerequisites

Before running this notebook, you must have ingested data into your Pinecone index:

1. **Legal data:** Run `Chapter08/Data_Ingestion.ipynb`
2. **Marketing data:** Run `Chapter09/Data_Ingestion_Marketing.ipynb`
   - **Critical:** Set `clear_index=False` in the marketing script to *append* data rather than overwrite, creating a multi-domain knowledge base.

Both `OPENAI_API_KEY` and `PINECONE_API_KEY` must be set in your Colab Secrets (the 🔑 key icon in the left sidebar).

---

## Architecture Overview

```
┌─────────────────────────────────────────────────────────────────┐
│                    GRADIO WEB INTERFACE                         │
│  ┌──────────────┐  ┌──────────────┐  ┌───────────────────────┐ │
│  │ Goal Presets │  │ Custom Input │  │ Moderation Toggle     │ │
│  │ (Dropdown)   │  │ (Text Area)  │  │ (Checkbox)            │ │
│  └──────┬───────┘  └──────┬───────┘  └──────────┬────────────┘ │
│         └─────────────────┴──────────────────────┘             │
│                            │                                    │
│                     [Run Engine]                                │
└────────────────────────────┼────────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────────┐
│                  UNIVERSAL CONTEXT ENGINE                       │
│                                                                 │
│  Pre-flight Moderation ──► Planner ──► Context Chaining        │
│                                 │                               │
│              ┌──────────────────┼──────────────────┐           │
│              ▼                  ▼                   ▼           │
│         Librarian           Researcher           Writer         │
│         Agent               Agent                Agent          │
│              │                  │                   │           │
│              └──────────────────┴──────────────────┘           │
│                                 │                               │
│                    Post-flight Moderation                       │
└─────────────────────────────────┼───────────────────────────────┘
                                  │
                                  ▼
┌─────────────────────────────────────────────────────────────────┐
│              GRADIO RESULTS PANEL (HTML Trace Dashboard)        │
│         Step Cards │ Token Metrics │ Final Orchestration Result │
└─────────────────────────────────────────────────────────────────┘
```

The Gradio layer is a thin presentation skin. The real intelligence lives in the engine modules downloaded from the repository.

---

# I. Initialization

## Step 1 — Download Engine Modules from GitHub

The engine is split across five Python modules hosted in the repository's `commons/engine/` directory. We download them fresh each session so this notebook always runs the latest version.

| Module | Responsibility |
|---|---|
| `utils.py` | Package installation and API client setup |
| `helpers.py` | Hardened LLM calls, embeddings, Pinecone queries, moderation |
| `agents.py` | Librarian, Researcher, and Writer specialist agents |
| `registry.py` | Agent capability registry (`AGENT_TOOLKIT`) |
| `engine.py` | Orchestrator — planner, context chaining, trace recorder |

In [1]:
print("Downloading engine modules from public repository...")

BASE = "https://raw.githubusercontent.com/Denis2054/Context-Engineering-for-Multi-Agent-Systems/main/commons/engine"
modules = ["utils.py", "helpers.py", "agents.py", "registry.py", "engine.py"]

for m in modules:
    !curl -Lfs {BASE}/{m} --output {m}
    print(f"  ✅ {m}")

print("\n✅ All engine modules downloaded successfully.")

  ✅ utils.py
  ✅ helpers.py
  ✅ agents.py
  ✅ registry.py
  ✅ engine.py

✅ All engine modules downloaded successfully.


---

## Step 2 — Install Dependencies and Initialize Clients

This step installs all required Python packages (`openai`, `pinecone`, `gradio`, `markdown`, etc.) and initializes the OpenAI and Pinecone API clients.

**How API keys are loaded:**
- In Google Colab, keys are read from **Colab Secrets** (`userdata.get()`)
- Locally, they are read from **environment variables** (`os.environ`)

This dual-path design means no secrets ever appear in notebook code.

In [2]:
import utils

# Install all required packages (openai, pinecone, gradio, markdown, etc.)
utils.install_dependencies()

# Initialize OpenAI and Pinecone clients using API keys from Colab Secrets / env vars
client, pc = utils.initialize_clients()

# Install Gradio (not part of the standard engine deps)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio", "-q"])
print("✅ Gradio installed.")

🚀 Installing required packages...
✅ All packages installed successfully.

🔑 Initializing API clients...
   - OpenAI client initialized.
   - Pinecone client initialized.
✅ Clients initialized successfully.
✅ Gradio installed.


---

# II. Context Engine — Core Imports

## Step 3 — Import Engine Modules

With the modules downloaded and dependencies installed, we import the engine stack.

This import order matters:
1. `helpers` must be imported before `agents` (agents call helpers internally)
2. `registry` must be imported before `engine` (engine queries the registry for capability descriptions)
3. `engine` is imported last so the `planner` function can be monkey-patched in the next step

In [3]:
import helpers        # LLM calls, embeddings, Pinecone search, moderation
import agents         # Librarian, Researcher, Writer agents
from registry import AGENT_TOOLKIT   # Agent capability registry
from engine import context_engine    # Main orchestrator

print("✅ All engine modules imported successfully.")

✅ All engine modules imported successfully.


---

## Step 4 — Render & Trace Dashboard

The dashboard renderer converts an engine `Trace` object into a rich HTML report. It surfaces:

- **Per-step cards** showing each agent invocation with collapsible input/output panels
- **Token metrics** (`📥 IN`, `📤 OUT`, `📉 SAVED`) for cost-efficiency monitoring
- **Execution time** for performance benchmarking
- **Final Orchestration Result** — the consolidated output after all agents have run

In the Gradio version, `render_trace_dashboard` returns an HTML string rather than calling `IPython.display`, so Gradio's `gr.HTML` component can display it directly inside the web interface.

In [4]:
import json
import html
import markdown as md_lib


def render_trace_dashboard(trace) -> str:
    """
    Converts an engine Trace object to a self-contained HTML string.
    Returns the HTML so Gradio can inject it into gr.HTML().
    """
    css = """
    <style>
        .dash { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
                background:#fff; border:3px solid #cbd5e0; border-radius:12px;
                padding:30px; color:#1a202c; }
        .hdr  { border-bottom:3px solid #2d3748; padding-bottom:20px; margin-bottom:25px;
                display:flex; justify-content:space-between; align-items:center; }
        .hdr h1 { margin:0; font-size:1.8rem; font-weight:900; color:#1a202c; }
        .hdr p  { margin:8px 0 0; color:#2d3748; font-size:1.1rem; font-style:italic; font-weight:600; }
        .badge  { padding:10px 20px; border-radius:8px; font-weight:900;
                  font-size:1rem; color:#fff; text-transform:uppercase; }
        .ok   { background:#22543d; } .fail { background:#742a2a; }
        .timer { margin-top:8px; font-size:1.1rem; font-weight:900;
                 background:#edf2f7; padding:5px 12px; border-radius:6px; color:#1a202c; }
        .step-card { background:#fff; border:2px solid #2d3748; border-radius:12px;
                     margin-bottom:22px; overflow:hidden;
                     box-shadow:0 4px 6px rgba(0,0,0,.05); }
        summary.shdr { padding:18px 20px; background:#f8fafc; cursor:pointer;
                       list-style:none; display:flex; align-items:center;
                       justify-content:space-between; border-bottom:2px solid #e2e8f0; }
        .abadge { background:#1a202c; color:#fff; padding:4px 12px; border-radius:6px;
                  font-size:.8rem; font-weight:900; text-transform:uppercase; margin-left:12px; }
        .pills  { display:flex; gap:10px; margin-top:10px; flex-wrap:wrap; }
        .pill   { background:#ebf4ff; color:#1a365d; padding:5px 12px; border-radius:8px;
                  border:2px solid #2b6cb0; font-size:.9rem; font-weight:900; }
        .saved  { background:#f0fff4; color:#1c4532; border-color:#2f855a; }
        .sbody  { padding:22px; }
        .dlabel { font-size:.95rem; text-transform:uppercase; font-weight:900;
                  border-left:4px solid #1a202c; padding-left:10px;
                  display:block; margin-bottom:10px; color:#1a202c; }
        .rendered { background:#fff; border:2px solid #e2e8f0; border-left:8px solid #2b6cb0;
                    padding:18px; color:#1a202c; line-height:1.7; font-size:1.05rem; font-weight:500; }
        .rendered h1,.rendered h2,.rendered h3,.rendered h4 {
            color:#1a202c !important; font-weight:900 !important;
            margin-top:1.2em; margin-bottom:.4em; }
        .rendered strong,.rendered b { font-weight:900; color:#000; }
        .jsonbox { background:#1a202c; color:#f7fafc; padding:18px; border-radius:10px;
                   font-family:monospace; font-size:.9rem; overflow-x:auto; }
        .final  { border:5px solid #22543d; background:#f0fff4; border-radius:12px;
                  padding:28px; margin-top:40px; color:#1a202c; }
        .final-title { color:#1c4532; font-size:1.4rem; font-weight:900;
                       text-transform:uppercase; letter-spacing:2px;
                       border-bottom:3px solid #22543d; padding-bottom:10px; margin-bottom:18px; }
        .open-lbl { font-weight:900; color:#fff; background:#1a202c;
                    padding:5px 12px; border-radius:6px; font-size:.75rem; }
    </style>
    """

    status_cls = "ok" if trace.status == "Success" else "fail"

    parts = [css, f"""
    <div class='dash'>
      <div class='hdr'>
        <div>
          <h1>Context Engine Trace</h1>
          <p>"{html.escape(trace.goal)}"</p>
        </div>
        <div style='text-align:right'>
          <span class='badge {status_cls}'>{trace.status}</span>
          <div class='timer'>⏱ {trace.duration:.2f}s</div>
        </div>
      </div>
      <h2 style='font-size:1.2rem;font-weight:900;text-transform:uppercase;
                 color:#1a202c;margin-bottom:20px;'>Execution Workflow</h2>
    """]

    def _md(text):
        return md_lib.markdown(str(text)) if text else "No content recorded."

    for step in trace.steps:
        try:
            resolved_ctx = json.dumps(step.get("resolved_context", {}), indent=2)
        except Exception:
            resolved_ctx = str(step.get("resolved_context", "N/A"))

        output_raw = step.get("output", "N/A")
        if isinstance(output_raw, dict):
            rendered = ""
            for key in ["summary", "answer_with_sources", "answer", "output", "content"]:
                if key in output_raw and isinstance(output_raw[key], str):
                    rendered = f"<div class='rendered'>{_md(output_raw[key])}</div>"
                    break
            if not rendered:
                rendered = f"<div class='jsonbox'>{html.escape(json.dumps(output_raw, indent=2))}</div>"
        else:
            rendered = f"<div class='rendered'>{_md(output_raw)}</div>"

        t_in   = step.get("tokens_in",  "??")
        t_out  = step.get("tokens_out", "??")
        t_save = step.get("tokens_saved", 0)
        save_pill = f"<span class='pill saved'>📉 SAVED: {t_save}</span>" if t_save else ""

        parts.append(f"""
        <details class='step-card' open>
          <summary class='shdr'>
            <div>
              <span style='font-weight:900;font-size:1.2rem;color:#1a202c;'>STEP {step['step']}</span>
              <span class='abadge'>{step['agent']}</span>
              <div class='pills'>
                <span class='pill'>📥 IN: {t_in}</span>
                <span class='pill'>📤 OUT: {t_out}</span>
                {save_pill}
              </div>
            </div>
            <span class='open-lbl'>OPEN LOGS</span>
          </summary>
          <div class='sbody'>
            <div style='margin-bottom:24px;'>
              <span class='dlabel'>Input Context (State)</span>
              <details>
                <summary style='font-size:.9rem;font-weight:900;color:#2b6cb0;cursor:pointer;margin-bottom:8px;'>
                  ▶ View Resolved Source Data
                </summary>
                <div class='jsonbox'>{html.escape(resolved_ctx)}</div>
              </details>
            </div>
            <span class='dlabel'>Agent Output</span>
            {rendered}
          </div>
        </details>
        """)

    if trace.final_output:
        fc = trace.final_output
        if isinstance(fc, dict):
            fc = fc.get("summary", fc.get("content", str(fc)))
        parts.append(f"""
        <div class='final'>
          <div class='final-title'>Final Orchestration Result</div>
          <div style='font-size:1.2rem;font-weight:700;line-height:1.8;'>{_md(fc)}</div>
        </div>
        """)

    parts.append("</div>")
    return "".join(parts)

---

## Step 5 — Engine Room: Robust Planner Patch + Execution Function

### Why the Planner Needs Patching

The default planner in `engine.py` works well for simple goals. For complex multi-step goals (e.g., *"Analyze the ChronoTech press release and summarize..."*), the LLM can hallucinate a slightly malformed JSON schema — placing agent arguments at the root of the step object instead of wrapping them inside an `"input"` key.

The **Robust Planner Patch** solves this by:
1. Displaying the exact required JSON schema directly in the system prompt
2. Using `json_mode=True` to force structured output
3. Adding a fallback that handles both `{"plan": [...]}` and bare list responses

This patch is applied via Python's module attribute assignment (`engine.planner = planner_robust_patch`) — a clean monkey-patch that requires no changes to the downloaded source files.

### execute_and_display

This is the main integration function called by the Gradio button handler. It:
1. Runs optional **pre-flight moderation** on the user's goal text
2. Calls `context_engine(...)` with the goal and configuration
3. Runs optional **post-flight moderation** on the AI output
4. Returns a rendered HTML trace string (or an error message) for Gradio to display

In [5]:
import engine
import json
import logging
import pprint
from helpers import call_llm_robust


# ── Robust Planner Patch ────────────────────────────────────────────────────

def planner_robust_patch(goal, capabilities, client, generation_model):
    """
    Enforces strict JSON schema in the planner prompt to prevent malformed plans
    on complex multi-step goals.
    """
    logging.info("[Planner-Patch] Generating strict-schema plan...")

    system_prompt = f"""
You are the strategic core of the Context Engine. Analyze the user's GOAL and create a step-by-step EXECUTION PLAN.

AVAILABLE CAPABILITIES
---
{capabilities}
---

INSTRUCTIONS:
1. Output MUST be a single JSON object with a "plan" key containing a list of step objects.
2. CRITICAL: Every step MUST strictly follow this schema:
   {{
      "step": <integer>,
      "agent": "<Agent Name>",
      "input": {{
          "<input_key>": "<input_value>"
      }}
   }}
   Agent arguments MUST be inside the "input" object, NOT at the root.
3. Use Context Chaining: write "$$STEP_N_OUTPUT$$" for values from previous steps.
"""
    try:
        raw = call_llm_robust(
            system_prompt, goal,
            client=client,
            generation_model=generation_model,
            json_mode=True
        )
        data = json.loads(raw)
        if "plan" not in data and isinstance(data, list):
            return data
        return data["plan"]
    except Exception as e:
        logging.error(f"Planner failed: {e}")
        raise


# Apply the patch
engine.planner = planner_robust_patch
print("✅ Engine patched with Robust Planner.")


# ── Main Execution Function ─────────────────────────────────────────────────

def execute_and_display(goal: str, config: dict, moderation_active: bool) -> str:
    """
    Runs the Context Engine and returns a rendered HTML trace string.
    Called by the Gradio button handler.
    """
    if not goal.strip():
        return "<p style='color:red;font-weight:bold;'>⚠️ Please enter a goal before running the engine.</p>"

    # Pre-flight moderation
    if moderation_active:
        report = helpers.helper_moderate_content(text_to_moderate=goal, client=client)
        if report.get("flagged"):
            return "<p style='color:red;font-weight:bold;'>🛑 Goal failed pre-flight moderation. Execution halted.</p>"

    try:
        result, trace = context_engine(goal, client=client, pc=pc, **config)
    except Exception as e:
        return f"<p style='color:red;font-weight:bold;'>🛑 Engine error: {html.escape(str(e))}</p>"

    # Post-flight moderation
    if result and moderation_active:
        text_to_check = json.dumps(result) if isinstance(result, (dict, list)) else str(result)
        report = helpers.helper_moderate_content(text_to_moderate=text_to_check, client=client)
        if report.get("flagged"):
            flagged_msg = "[Content flagged by moderation policy and redacted.]"
            if trace:
                trace.final_output = flagged_msg

    if trace:
        return render_trace_dashboard(trace)

    return "<p style='color:red;'>Engine failed to produce a trace.</p>"


print("✅ Engine Room ready.")

✅ Engine patched with Robust Planner.
✅ Engine Room ready.


---

## Step 6 — Control Deck Configuration

The `config` dictionary is the **Control Deck** of the Universal Context Engine. It wires the engine to your specific infrastructure:

| Parameter | Purpose |
|---|---|
| `index_name` | Name of the Pinecone index holding the multi-domain knowledge base |
| `generation_model` | The LLM used for reasoning (planning, summarizing, writing) |
| `embedding_model` | The model used to encode queries for vector search |
| `namespace_context` | Pinecone namespace for operational instructions and context rules |
| `namespace_knowledge` | Pinecone namespace for domain documents (legal, marketing, etc.) |

**Important:** The namespaces separate *how* the engine should behave (`ContextLibrary`) from *what* it knows (`KnowledgeStore`). This separation is fundamental to the Glass Box design — swapping one namespace changes the engine's persona without touching any code.

In [6]:
import html  # needed by execute_and_display

# ── Control Deck Configuration ──────────────────────────────────────────────
# Modify these values to point to your own Pinecone index and preferred models.
config = {
    "index_name":          "genai-mas-mcp-ch3",
    "generation_model":    "gpt-5.1",
    "embedding_model":     "text-embedding-3-small",
    "namespace_context":   "ContextLibrary",
    "namespace_knowledge": "KnowledgeStore",
}

print("✅ Control Deck configuration loaded.")
print(f"   Index:            {config['index_name']}")
print(f"   Generation model: {config['generation_model']}")
print(f"   Embedding model:  {config['embedding_model']}")

✅ Control Deck configuration loaded.
   Index:            genai-mas-mcp-ch3
   Generation model: gpt-5.1
   Embedding model:  text-embedding-3-small


---

# III. Gradio Standalone UI

## Step 7 — Build and Launch the Web Application

This cell constructs the Gradio interface and launches it. The design follows a three-zone layout:

```
┌─────────────────────────────────────────────────────────────┐
│  ZONE 1 — INPUT PANEL                                       │
│  Preset selector  │  Goal text area  │  Moderation toggle  │
│  [Load Preset]    │  [Clear]         │  [Run Engine ▶]     │
├─────────────────────────────────────────────────────────────┤
│  ZONE 2 — STATUS BAR                                        │
│  Shows "Running..." during execution, clears on completion  │
├─────────────────────────────────────────────────────────────┤
│  ZONE 3 — RESULTS PANEL                                     │
│  HTML Trace Dashboard with step cards and final output      │
└─────────────────────────────────────────────────────────────┘
```

### Gradio Concepts Used

| Gradio Component | Role |
|---|---|
| `gr.Dropdown` | Preset goal selector |
| `gr.Textbox` | Editable goal input |
| `gr.Checkbox` | Moderation toggle |
| `gr.Button` | Triggers engine execution |
| `gr.HTML` | Renders the trace dashboard |
| `gr.Markdown` | Section headers and status messages |
| `gr.Blocks` | Layout container with custom CSS theming |

### Public URL

When `share=True`, Gradio creates a temporary public tunnel URL (valid for 72 hours). This allows you to share a live demo with colleagues who do not have Colab access. For permanent deployment, export this notebook to a Python script and deploy it on Hugging Face Spaces or any Docker host.

### Preset Goals

Six presets are included, organized by domain:

**Marketing:**
- Quantum Drive product query (simple RAG retrieval)
- ChronoTech press release analysis with citations (multi-step RAG)
- Persuasive pitch writing (creative generation from retrieved context)

**Legal:**
- NDA retrieval and summary (two-step: retrieve then summarize)
- Service Agreement high-fidelity RAG with citations
- Sanitization limit test (triggers the Researcher's sanitizer guardrail)

In [ ]:
import gradio as gr

# ── Preset Goals ─────────────────────────────────────────────────────────────
PRESETS = {
    "── Select a preset goal ──": "",
    "[Marketing] 1. Quantum Drive Query": \
        "Summarize the key points of the QuantumDrive",
    "[Marketing] 2. ChronoTech Press Release Analysis (with citations)": \
        "Analyze the ChronoTech press release and summarize their core product "
        "messaging and value proposition. Please cite your sources.",
    "[Marketing] 3. Persuasive Brand Pitch": \
        "Write a persuasive pitch on our brand tone and voice guide",
    "[Legal] 4. NDA Retrieval & Summary": \
        "First, retrieve the content of the Non-Disclosure Agreement (NDA) from "
        "the knowledge base. Then, summarize its key points.",
    "[Legal] 5. Service Agreement — High-Fidelity RAG (with citations)": \
        "What are the key confidentiality obligations in the Service Agreement v1, "
        "and what is the termination notice period? Please cite your sources.",
    "[Legal] 6. Sanitization Limit Test": \
        "What did Mr. Smith advise his client regarding the assets?",
}


# ── Gradio Handler Functions ──────────────────────────────────────────────────

def load_preset(preset_name: str):
    """Fills the goal textbox when a preset is selected from the dropdown."""
    return PRESETS.get(preset_name, "")


def run_engine(goal: str, moderation: bool):
    """
    Gradio button handler. Yields intermediate status then the final HTML trace.
    Using yield turns this into a streaming generator so Gradio can show a
    'Running...' message before the (potentially slow) engine response arrives.
    """
    if not goal.strip():
        yield (
            "",
            "<p style='color:#c05621;font-weight:bold;'>"
            "⚠️ Please enter a goal before running the engine.</p>"
        )
        return

    # Show a running indicator immediately
    yield (
        "⏳ **Context Engine running...** Multi-agent orchestration in progress.",
        ""
    )

    html_result = execute_and_display(goal, config, moderation)

    yield ("", html_result)   # clear status, show result


# ── Custom CSS ────────────────────────────────────────────────────────────────
CUSTOM_CSS = """
#run-btn { background: #22543d !important; color: white !important;
           font-size: 1.1rem !important; font-weight: 900 !important;
           border-radius: 8px !important; }
#run-btn:hover { background: #276749 !important; }
#clear-btn { font-weight: 700 !important; border-radius: 8px !important; }
.gr-form { border-radius: 12px !important; }
.gradio-container { max-width: 1200px !important; margin: auto !important; }
"""


# ── Interface Definition ──────────────────────────────────────────────────────
with gr.Blocks(
    title="Universal Context Engine",
    theme=gr.themes.Default(primary_hue="green", neutral_hue="slate"),
    css=CUSTOM_CSS
) as demo:

    # ── Header ────────────────────────────────────────────────────────────────
    gr.Markdown("""
    # 🧠 Universal Context Engine
    ### *Context Engineering for Multi-Agent Systems* — Standalone Web UI

    A **domain-agnostic Glass Box** multi-agent reasoning engine.
    Zero hard-coded business rules. Behavior is driven entirely by **context**.

    Select a preset goal or type your own below, then click **Run Engine**.
    """)

    gr.Markdown("---")

    # ── Input Zone ────────────────────────────────────────────────────────────
    with gr.Row():
        with gr.Column(scale=3):
            gr.Markdown("### Control Deck")

            preset_dd = gr.Dropdown(
                label="Load Preset Goal",
                choices=list(PRESETS.keys()),
                value="── Select a preset goal ──",
                interactive=True,
                info="Select a preset to auto-fill the goal box, or type directly."
            )

            goal_box = gr.Textbox(
                label="Goal (edit freely or type your own)",
                placeholder="Type your high-level goal here...",
                lines=4,
                interactive=True,
            )

            moderation_cb = gr.Checkbox(
                label="Enable Moderation Guardrails",
                value=False,
                info="Pre-flight checks goal text; post-flight checks AI output."
            )

        with gr.Column(scale=1):
            gr.Markdown("### Actions")
            run_btn   = gr.Button("▶ Run Engine",   variant="primary",   elem_id="run-btn")
            clear_btn = gr.Button("✕ Clear",         variant="secondary", elem_id="clear-btn")

            gr.Markdown("""
            **Domains:** Legal · Marketing

            **Agents:** Librarian · Researcher · Writer

            **Guardrails:** Pre-flight · Post-flight

            **Analytics:** Token in/out, execution time
            """)

    gr.Markdown("---")

    # ── Status Bar ────────────────────────────────────────────────────────────
    status_md = gr.Markdown("")

    # ── Results Zone ──────────────────────────────────────────────────────────
    gr.Markdown("### Trace Dashboard")
    result_html = gr.HTML(label="Engine Output")

    # ── Glossary / Reference ──────────────────────────────────────────────────
    with gr.Accordion("📚 Reference: Architecture Glossary", open=False):
        gr.Markdown("""
        | Term | Definition |
        |---|---|
        | **Glass Box** | An engine with fully auditable, transparent reasoning steps — no hidden logic |
        | **Control Deck** | The configuration layer that sets the engine's goal, model, and data namespaces |
        | **Context Chaining** | Passing the output of one agent step as input to the next via `$$STEP_N_OUTPUT$$` tokens |
        | **High-Fidelity RAG** | Retrieval-Augmented Generation with source citations, enabling verifiable answers |
        | **ContextLibrary** | Pinecone namespace holding operational instructions and domain-specific context rules |
        | **KnowledgeStore** | Pinecone namespace holding the actual documents (legal, marketing, etc.) |
        | **Librarian Agent** | Retrieves context instructions from `ContextLibrary` to configure downstream agents |
        | **Researcher Agent** | Performs RAG queries against `KnowledgeStore`, enforces sanitization on retrieved chunks |
        | **Writer Agent** | Synthesizes a final coherent response from all previous agents' outputs |
        | **Pre-flight Moderation** | Safety check on user input *before* the engine runs |
        | **Post-flight Moderation** | Safety check on AI output *before* it is displayed to the user |
        | **Token Analytics** | Per-step tracking of input/output tokens for cost-efficiency monitoring |
        """)

    with gr.Accordion("💡 How to Use This UI", open=False):
        gr.Markdown("""
        ### Quick Start

        1. **Select a preset** from the dropdown — it auto-fills the goal box
        2. **Edit the goal** freely — the text box is fully editable
        3. Optionally **enable Moderation Guardrails**
        4. Click **▶ Run Engine** and wait for the trace dashboard

        ### Understanding the Results

        - Each **step card** shows one agent invocation. Click the header to collapse/expand.
        - The **▶ View Resolved Source Data** link expands the raw JSON context that was passed to the agent.
        - **Token pills** (📥 IN / 📤 OUT) show per-step model usage.
        - The **Final Orchestration Result** (green card at the bottom) is the engine's consolidated answer.

        ### Proving Domain-Agnosticism

        Run preset **#1 or #2** (Marketing), then run **#4 or #5** (Legal). Notice that:
        - The **same three agents** are used in both domains
        - The **same planner logic** generates the execution plan
        - The **same Pinecone index** holds both domains' documents
        - Only the **goal text** drives the different behavior

        This is the core thesis of *Context Engineering for Multi-Agent Systems*.
        """)

    # ── Event Bindings ────────────────────────────────────────────────────────
    preset_dd.change(
        fn=load_preset,
        inputs=[preset_dd],
        outputs=[goal_box]
    )

    run_btn.click(
        fn=run_engine,
        inputs=[goal_box, moderation_cb],
        outputs=[status_md, result_html]
    )

    clear_btn.click(
        fn=lambda: ("── Select a preset goal ──", "", False, "", ""),
        outputs=[preset_dd, goal_box, moderation_cb, status_md, result_html]
    )


# ── Launch ────────────────────────────────────────────────────────────────────
print("Launching Gradio app...")
demo.launch(
    share=True,         # Creates a public tunnel URL — share with colleagues!
    debug=False,        # Set True to see Python tracebacks in the browser
    quiet=True,
)

---

# IV. Deployment Guide

## Running Locally (Outside Colab)

To run this as a persistent local web server:

```bash
# 1. Export the notebook to a Python script
jupyter nbconvert --to script Universal_Context_Engine_Gradio_UI.ipynb

# 2. Set environment variables
export OPENAI_API_KEY="sk-..."
export PINECONE_API_KEY="..."

# 3. Run the script
python Universal_Context_Engine_Gradio_UI.py
```

The app will be available at `http://localhost:7860`.

---

## Deploying to Hugging Face Spaces

Gradio apps deploy natively to [Hugging Face Spaces](https://huggingface.co/spaces) — the fastest path to a permanent public URL.

```bash
# 1. Create a new Space at https://huggingface.co/new-space
#    SDK: Gradio | Hardware: CPU Basic (free)

# 2. Clone your Space repo
git clone https://huggingface.co/spaces/YOUR_USER/YOUR_SPACE
cd YOUR_SPACE

# 3. Copy the exported script as app.py
cp Universal_Context_Engine_Gradio_UI.py app.py

# 4. Add a requirements.txt
echo -e "openai\npinecone-client\ngradio\nmarkdown" > requirements.txt

# 5. Set secrets in the Space settings:
#    OPENAI_API_KEY, PINECONE_API_KEY

# 6. Push
git add . && git commit -m "Deploy Universal Context Engine" && git push
```

Your app will be live at `https://huggingface.co/spaces/YOUR_USER/YOUR_SPACE` within minutes.

---

## Important Deployment Notes

- **Change `share=False`** in `demo.launch()` for production deployments (Spaces handles routing automatically)
- **Remove the `!curl` download cells** and commit the engine modules alongside `app.py` so the Space has all files on startup
- The `config` dictionary (`index_name`, `generation_model`, etc.) can be exposed as **Gradio Settings** for operator-level control without code changes

---

## Further Reading

- *Context Engineering for Multi-Agent Systems* — Denis Rothman
- [Gradio Documentation](https://www.gradio.app/docs)
- [Pinecone Documentation](https://docs.pinecone.io)
- [OpenAI API Reference](https://platform.openai.com/docs)
- [Hugging Face Spaces Guide](https://huggingface.co/docs/hub/spaces)